# Latent Dirichlet Allocation (topic modeling)

_Artificial Intelligence — more_

**Every document is a blend of hidden topics. Each topic is a favorite set of words.**

Latent Dirichlet Allocation (LDA) discovers hidden topics in a collection of documents, with no labels.

     Each document is a mixture of topics, described by proportions $\theta$. Each topic is a distribution over words, $\phi$.

---

This notebook builds a collapsed-Gibbs LDA sampler from scratch, one step at a time. Run each cell, read the note above it, and watch the recovered topics separate the words. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

We'll recover hidden topics from four tiny documents using **collapsed Gibbs sampling**. We build it in three steps: (1) the documents and count tables, (2) the Gibbs sweep that reassigns each word to a topic, (3) reading off the topic-word distributions.

### Step 1 — Documents, counts, and a random start

Each document is a list of word ids over a vocabulary of 4 words. Documents 0 and 1 lean on words `{0, 1}`; documents 2 and 3 lean on words `{2, 3}` — so a good model should find two topics. We assign every word a random topic `z`, then tally two count tables: `ndk` (how many words in each document belong to each topic) and `nkw` (how many times each word is assigned to each topic).

In [ ]:
rng = np.random.default_rng(0)

# Vocab 0..3. Docs 0,1 favor words {0,1}; docs 2,3 favor words {2,3}.
docs = [[0, 1, 0, 1], [1, 0, 1, 0], [2, 3, 2, 3], [3, 2, 3, 2]]
V = 4              # vocabulary size
K = 2              # number of topics
alpha = 0.1        # Dirichlet prior on doc-topic mix
beta = 0.1         # Dirichlet prior on topic-word mix

# Start every word at a random topic.
z = [rng.integers(0, K, len(d)).tolist() for d in docs]

ndk = np.zeros((len(docs), K))    # doc-topic counts
nkw = np.zeros((K, V))            # topic-word counts
for d, doc in enumerate(docs):
    for i, w in enumerate(doc):
        topic = z[d][i]
        ndk[d, topic] += 1
        nkw[topic, w] += 1

### Step 2 — Reassign each word's topic with collapsed Gibbs

On each sweep we visit every word, **remove** it from the counts, then draw a new topic with probability proportional to `(doc's affinity for the topic) x (topic's affinity for this word)`. After sampling we **add** the word back under its new topic. Repeating this 300 times lets the assignments settle so that co-occurring words drift into the same topic.

In [ ]:
for sweep in range(300):              # collapsed Gibbs sampling
    for d, doc in enumerate(docs):
        for i, w in enumerate(doc):
            # Remove word i from the counts before resampling its topic.
            k = z[d][i]
            ndk[d, k] -= 1
            nkw[k, w] -= 1

            # P(topic) proportional to doc-topic affinity times topic-word affinity.
            p = (ndk[d] + alpha) * (nkw[:, w] + beta) / (nkw.sum(1) + V * beta)
            k = rng.choice(K, p=p / p.sum())

            # Add the word back under its newly sampled topic.
            z[d][i] = k
            ndk[d, k] += 1
            nkw[k, w] += 1

### Step 3 — Read off the topic-word distributions

`phi[k]` is topic `k`'s distribution over the vocabulary, smoothed by `beta`. Each row should now concentrate on one half of the vocabulary — confirming the sampler split words `{0, 1}` from words `{2, 3}` into two distinct topics.

In [ ]:
phi = (nkw + beta) / (nkw.sum(1, keepdims=True) + V * beta)
print("topic-word distributions (rows = topics):")
print(np.round(phi, 2))

## Visualize the data & results

_Six news headlines, two hidden topics: does collapsed-Gibbs LDA separate the Sports words from the Finance words?_

### Step 1 — Encode six headlines over a real vocabulary

We use six short headlines drawn from two themes: Sports words (`game`, `team`, `win`) and Finance words (`market`, `stock`, `bank`). We map each word to an integer id so the documents become lists of ids, exactly like the toy example above.

In [ ]:
rng = np.random.default_rng(1)

# Six real-style news headlines over two topics.
vocab = ["game", "team", "win", "market", "stock", "bank"]
raw = [["team", "win", "game", "game"], ["game", "team", "win", "team"],
       ["win", "game", "game", "team"], ["market", "stock", "bank", "stock"],
       ["stock", "market", "bank", "market"], ["bank", "stock", "market", "stock"]]

wi = {w: i for i, w in enumerate(vocab)}      # word -> id lookup
docs = [[wi[w] for w in h] for h in raw]
V = len(vocab)
K = 2
a = 0.1        # doc-topic prior
b = 0.1        # topic-word prior

### Step 2 — Run the same collapsed-Gibbs sampler

This is the identical remove / resample / add loop from the reference, run for 500 sweeps over the headlines. We initialize the topic assignments randomly and build the same `ndk` and `nkw` count tables before sweeping.

In [ ]:
z = [rng.integers(0, K, len(d)).tolist() for d in docs]
ndk = np.zeros((len(docs), K))
nkw = np.zeros((K, V))
for d, doc in enumerate(docs):
    for i, w in enumerate(doc):
        topic = z[d][i]
        ndk[d, topic] += 1
        nkw[topic, w] += 1

for sweep in range(500):                     # collapsed Gibbs sampling
    for d, doc in enumerate(docs):
        for i, w in enumerate(doc):
            k = z[d][i]
            ndk[d, k] -= 1
            nkw[k, w] -= 1

            p = (ndk[d] + a) * (nkw[:, w] + b) / (nkw.sum(1) + V * b)
            k = rng.choice(K, p=p / p.sum())

            z[d][i] = k
            ndk[d, k] += 1
            nkw[k, w] += 1

### Step 3 — Plot the recovered topic-word heatmap

We compute the smoothed `phi` distributions, then flip the rows if needed so row 0 is the Sports topic (topic labels from unsupervised LDA are arbitrary). The heatmap should show each topic lighting up its own three words, confirming the Sports and Finance vocabularies were separated.

In [ ]:
phi = (nkw + b) / (nkw.sum(1, keepdims=True) + V * b)

# Topic order is arbitrary; flip so row 0 is the Sports topic.
if phi[1, wi["game"]] > phi[0, wi["game"]]:
    phi = phi[::-1]

fig, ax = plt.subplots()
im = ax.imshow(phi, cmap="viridis")
for r in range(K):
    for c in range(V):
        ax.text(c, r, round(phi[r, c], 3), ha="center", va="center", color="w")
ax.set_xticks(range(V))
ax.set_xticklabels(vocab)
ax.set_yticks(range(K))
ax.set_yticklabels(["Topic: Sports", "Topic: Finance"])
ax.set_title("Recovered topic-word distributions from 6 news headlines")
fig.colorbar(im, ax=ax)
plt.show()